Author: Yicheng Rui (TDLI)

Email: ruiyicheng@sjtu.edu.cn

The image we captured is for a single transit. In this notebook, we will first fit the single transit then compare the results with the previous results to see whether there is a transit-timing variation (TTV).

# Read the data and show the light curve

In [ ]:
#read the data
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
AIJ_res = pd.read_csv('./input/light_curve/light_curve.csv')
JD_utc = np.array(AIJ_res['JD_UTC']) # JD_UTC is the mid of the exposure (How do I know that?)
rel_flux = np.array(AIJ_res['rel_flux_T1'])
e_rel_flux = np.array(AIJ_res['rel_flux_err_T1'])
fig ,ax =plt.subplots(1,1,figsize=(16,8))
ax.ticklabel_format(useOffset=False, style='plain')
plt.errorbar(JD_utc,rel_flux,marker = '.',yerr = e_rel_flux)#, ls='none')
plt.xlabel('JD in UTC')
plt.ylabel('relative flux')

# Fit the light curve
In this section, we use a quartic polynomial model to fit this single transit. Flux of the model $\hat{F}$ can be written as
\begin{equation}
{\hat{F}(t;A_0,A_2,A_4,t_m,d)} = \left\{\begin{aligned}
A_0+A_2(t-t_m)^2+A_4(t-t_m)^4&\qquad (t_m-\frac{d}{2}\le t\le t_m+\frac{d}{2})\\
A_0+A_2(\frac{d}{2})^2+A_4(\frac{d}{2})^4&\qquad {\rm otherwise}
\end{aligned}\right.,
\end{equation}
where $A_0,A_2,A_4,t_m,d$ are parameters to be estimated. $t_m$ is the mid-transit time and $d$ is the transit duration. Once $t_m$ and $d$ are fixed, the remaining parameters are linear thus can be easily optimized by linear regression. The linearized model can be written as
\begin{equation}
{\vec{\hat{F}}(t_1,\cdots,t_n;A_0,A_2,A_4|t_m,d)} = \begin{pmatrix}1&\mathbb{1}(t_1\in T)(t_1-t_m)^2+(1-\mathbb{1}(t_1\in T))(\frac{d}{2})^2&\mathbb{1}(t_1\in T)(t_1-t_m)^4+(1-\mathbb{1}(t_1\in T))(\frac{d}{2})^4\\
1&\mathbb{1}(t_2\in T)(t_2-t_m)^2+(1-\mathbb{1}(t_2\in T))(\frac{d}{2})^2&\mathbb{1}(t_2\in T)(t_2-t_m)^4+(1-\mathbb{1}(t_2\in T))(\frac{d}{2})^4\\
\vdots&\vdots&\vdots\\
1&\mathbb{1}(t_n\in T)(t_n-t_m)^2+(1-\mathbb{1}(t_n\in T))(\frac{d}{2})^2&\mathbb{1}(t_n\in T)(t_n-t_m)^4+(1-\mathbb{1}(t_n\in T))(\frac{d}{2})^4
\end{pmatrix}\begin{pmatrix}A_0\\A_2\\A_4\end{pmatrix} = X\theta,
\end{equation}
where $\mathbb{1}(\cdot)$ is indicator function; $T=[t_m-\frac{d}{2},t_m+\frac{d}{2}]$; $X$ is the design matrix; $\theta = (A_0,A_2,A_4)^T$ is the linear parameters. The loglikehood of the linearized model is

\begin{equation}
    \log L = -\frac{1}{2}(Y-X\theta)^T\Sigma^{-1}(Y-X\theta),
\end{equation}
where $Y$ is the relative flux; $\Sigma$ is the covarient matrix of the data. Now we ignore the correlation between differnent data points. $\Sigma$ is thus a diagnal matrix which elements are the square of the flux uncertainty. The maximun-likelihood solution and its covarience are

\begin{equation}
\theta = (X^T \Sigma^{-1} X)^{-1} X^T \Sigma^{-1} Y;\quad \Sigma_\theta = (X^T \Sigma^{-1} X)^{-1}.
\end{equation}
The proof is left as an exercise.

In this hands-on, we would like to traverse $x_m$ and $\frac{d}{2}$ to find all the parameters that minimize the likelihood.

In [ ]:
def loglikelihood(X,theta,Y,Sigma):
    return -0.5*(Y-X.dot(theta)).T.dot(np.linalg.inv(Sigma)).dot(Y-X.dot(theta))

In [ ]:
JD_utc_max = np.max(JD_utc)
JD_utc_min = np.min(JD_utc)
time_normalize = (JD_utc-JD_utc_min)/(JD_utc_max-JD_utc_min) # Why do I need this normalization?
Y = rel_flux.reshape(-1,1)                                   # reshape into a column vector
t = time_normalize.reshape(-1,1)                             # reshape into a column vector
d_try = np.linspace(0,1,70)
t_m_try = np.linspace(0,1,70)
Sigma = np.diag(e_rel_flux**2)
maximum_ll = -np.inf
for d in d_try:
    for t_m in t_m_try:
        in_transit = (t>=t_m-d/2)&(t<=t_m+d/2)
        if np.sum(in_transit)>3:
            X = np.hstack([np.ones(t.shape),in_transit*(t-t_m)**2+(1-in_transit)*(d/2)**2,in_transit*(t-t_m)**4+(1-in_transit)*(d/2)**4])
            theta = np.linalg.inv(X.T.dot(np.linalg.inv(Sigma)).dot(X)).dot(X.T).dot(np.linalg.inv(Sigma)).dot(Y)
            ll = loglikelihood(X,theta,Y,Sigma)
            if ll > maximum_ll:
                maximum_ll = ll
                best_d = d
                best_t_m = t_m
                best_theta = theta

In [ ]:
print('best d =' , best_d)
print('best t_m =' , best_t_m)
print('best theta =\n' , best_theta)

In [ ]:
# Show the fit result
d = best_d
t_m = best_t_m
in_transit = (t>=t_m-d/2)&(t<=t_m+d/2)
X = np.hstack([np.ones(t.shape),in_transit*(t-t_m)**2+(1-in_transit)*(d/2)**2,in_transit*(t-t_m)**4+(1-in_transit)*(d/2)**4])
Y_hat = np.squeeze(X.dot(best_theta))

fig ,ax =plt.subplots(1,1,figsize=(16,8))
ax.ticklabel_format(useOffset=False, style='plain')
plt.errorbar(time_normalize,rel_flux,marker = '.',yerr = e_rel_flux,label = 'raw data', ls='none')
plt.plot(time_normalize,Y_hat,'--',label = 'fit result')
plt.xlabel('normalized time')
plt.ylabel('relative flux')
plt.legend()

# Obtain the uncertainty of mid-transit time using MCMC

Now we have a rough estimation about this single transit. To get the uncertainty of mid-transit time and transit duration, we need to use Markov chain Monte Carlo (MCMC) to sample the parameters. Emcee is used to sample the parameter.


In [ ]:
def loglikelihood_mcmc(param,Y = Y,Sigma = Sigma,t = t):
    d,t_m,A0,A2,A4 = np.squeeze(param)
    # if A2<0:
    #     return -np.inf
    theta = np.array([A0,A2,A4]).reshape(-1,1)
    in_transit = (t>=t_m-d/2)&(t<=t_m+d/2)
    X = np.hstack([np.ones(t.shape),in_transit*(t-t_m)**2+(1-in_transit)*(d/2)**2,in_transit*(t-t_m)**4+(1-in_transit)*(d/2)**4])
    return -0.5*(Y-X.dot(theta)).T.dot(np.linalg.inv(Sigma)).dot(Y-X.dot(theta))

In [ ]:
import emcee
#print(best_d,np.squeeze(best_theta[1]))
param_ini = np.array([best_d,best_t_m,best_theta[0][0],best_theta[1][0],best_theta[2][0]]).reshape(1,-1)
pos = param_ini + 1e-2 * np.random.randn(32, 5)
nwalkers, ndim = pos.shape

sampler = emcee.EnsembleSampler(
    nwalkers, ndim, loglikelihood_mcmc
)
ret = sampler.run_mcmc(pos, 5000, progress=True)

In [ ]:
# Show the MCMC results
import corner
flat_samples = sampler.get_chain(discard=1000, thin=15, flat=True)
print(flat_samples.shape)
fig = corner.corner(
    flat_samples, labels=['d',r'$t_m$',r'$A_0$',r'$A_2$',r'$A_4$']
)

# Problem: There exists samples that satisfy $A_2<0$, which is unnatural. Why? How to fix that?

In [ ]:
#Obtain the error bar of t
t_m_lower,t_m_median,t_m_upper = np.percentile(flat_samples[:, 1], [16, 50, 84])
t_m_err = (t_m_upper-t_m_lower)/2
t_m_median_jd_UTC = t_m_median*(JD_utc_max-JD_utc_min)+JD_utc_min #Convert normalized time back into JD
e_t_m_median_jd_UTC = t_m_err*(JD_utc_max-JD_utc_min)
print(t_m_median_jd_UTC,e_t_m_median_jd_UTC)

Since the time is utc, to compare with other single transit, we need to use a universal time scale that is irrelevant to the place of Earth. We need to convert the UTC time into TDB (Barycentric dynamic time) at barycenter of the solar system to compare with former results.

In [ ]:
from astropy import time, coordinates as coord, units as u
wasp11b_coord = coord.SkyCoord("03:09:28.5433589520","+30:40:24.862993356",
                        unit=(u.hourangle, u.deg), frame='icrs')
MGO = coord.EarthLocation.from_geodetic(lon = (121+36/60+21.8/3600)*u.deg, lat = (31+10/60+07.7/3600)*u.deg,height = 30)
time_mtt = time.Time(t_m_median_jd_UTC, format='jd',scale='utc', location=MGO) 
ltt_bary = time_mtt.light_travel_time(wasp11b_coord)
time_barycentre = float((time_mtt.tdb + ltt_bary).jd)
# Show the difference
print(time_mtt.tdb)
print(3600*24*(float(time_mtt.jd)-float(time_mtt.tdb.jd)))
print(time_barycentre,3600*24*ltt_bary)
print(3600*24*(time_barycentre-t_m_median_jd_UTC))

# Compare with history data
The history mid-transit time is downloaded at https://osf.io/p298n/.

The ephemerides of transit targets are available at https://vizier.cds.unistra.fr/viz-bin/VizieR-3?-source=J/ApJS/265/4/table7.

In [ ]:
#read ephemerides data for target star
ephemerides = pd.read_csv('./input/mid_transit_time/ephemerides.csv',sep = '|',comment = '#')
ephemerides['Planet'] = ephemerides['Planet'].str.strip()
WASP_11_eph = ephemerides[ephemerides['Planet']=='WASP-11b']
T0 = float(WASP_11_eph['T0'].iloc[0])
e_T0 = float(WASP_11_eph['e_T0'].iloc[0])
period = float(WASP_11_eph['Per'].iloc[0])
e_period = float(WASP_11_eph['e_Per'].iloc[0])
print(T0,e_T0,period,e_period)

In [ ]:
# Compute observe-compute (O-C) for this observation
O_C_this = ((((time_barycentre-T0)/period)-((time_barycentre-T0)//period)+0.5)%1-0.5)*period*24*3600 #We use unit in seconds
print(O_C_this)

In [ ]:
# Compute previous O-C
previous_data = pd.read_csv('./input/mid_transit_time/WASP-11b.csv',sep = '\t',comment = '#')
T_mid = np.array(previous_data.iloc[:,0],dtype = np.float64)
e_T_mid = np.array(previous_data.iloc[:,1],dtype = np.float64)
O_C_ephemeride = ((((T_mid-T0)/period)-((T_mid-T0)//period)+0.5)%1-0.5)*period*24*3600
O_C_ephemeride

In [ ]:
# Show the results
fig ,ax =plt.subplots(1,1,figsize=(16,8))
ax.ticklabel_format(useOffset=False, style='plain')
#plt.errorbar(time_normalize,rel_flux,marker = '.',yerr = e_rel_flux,label = 'raw data', ls='none')
plt.plot([np.min(T_mid),time_barycentre],[0,0],'--',label = 'O-C = 0')
plt.xlabel('BJD')
plt.ylabel('O-C [s]')
plt.errorbar(T_mid,O_C_ephemeride,yerr = e_T_mid*24*3600, marker = '.',ls='none',label = 'history data')
plt.errorbar([time_barycentre],[O_C_this],yerr = e_t_m_median_jd_UTC*24*3600, marker = '.',ls='none',label = 'This measurement')
plt.legend()

# Detecting periodic signals in O-C data
We want to investigate whether there is periodic signal in the mid transit time. What we want to do is to compare two models. The first model is the baseline model, which is a constant combined with linear trend (Why?). The log likelihood can be written as

\begin{equation}
\log L_{\rm baseline} = -\frac{1}{2}(Y-C \vec{1}-K \vec{t})^T\Sigma^{-1}(Y-C \vec{1}-K \vec{t}),
\end{equation}
where $Y$ is the O-C value; $\Sigma$ is a diagnal matrix whose elements are the square of uncertainty of O-C; \vec{1} is a column vector whose all elements are 1. The log likelihood of the periodic model can be written as

\begin{equation}
\log L_{\rm periodic} = -\frac{1}{2}(Y-A\cos(\frac{2\pi}{P} \vec{t})-B\sin(\frac{2\pi}{P} \vec{t})-C \vec{1}-K \vec{t})^T\Sigma^{-1}(Y-A\cos(\frac{2\pi}{P} \vec{t})-B\sin(\frac{2\pi}{P} \vec{t})-C \vec{1}-K \vec{t}),
\end{equation}
where $P$ is the period of the signal. We need to traverse $P$ to see which period is contained in the data. This technique is called periodogram. 

To check whether there is significant periodic signal for a given $P$, we need to use Bayes factor (BF). BF between two models can be estimated using

\begin{equation}
\text{lnBF} = \frac{\rm BIC_{\rm baseline}-BIC_{\rm periodic}}{2};\qquad {\rm BIC}=k \log n -2\log\hat{L},
\end{equation}
where BIC is Bayesian information criterion; $k$ is the number of the parameter; $\hat{L}$ is the maximized likelihood of a model; $n$ is the number of data points. The derivation of BIC is available in wiki https://en.wikipedia.org/wiki/Bayesian_information_criterion. lnBF = 5 is often used as a threshold for significant signal. 


In [ ]:
def lnBF(P,time,O_C,e_O_C):
    def max_ll(X,Y,invS):
        return np.linalg.inv((X.T.dot(invS).dot(X))).dot(X.T).dot(Y)
    def ll(X,Y,invS,theta):
        return -0.5*(Y-X.dot(theta)).T.dot(invS).dot(Y-X.dot(theta))
    Y = O_C.reshape(-1,1)
    Sigma = np.diag(e_O_C**2)
    inv_Sigma = np.linalg.inv(Sigma)
    t = time.reshape(-1,1)
    Xbase = np.hstack([np.ones(t.shape),t])
    Xperiodic = np.hstack([np.cos(2*np.pi/P*t),np.sin(2*np.pi/P*t),np.ones(t.shape),t])
    theta_base = max_ll(Xbase,Y,inv_Sigma)
    theta_periodic = max_ll(Xperiodic,Y,inv_Sigma)
    ll_base = ll(Xbase,Y,inv_Sigma,theta_base)
    ll_periodic = ll(Xperiodic,Y,inv_Sigma,theta_periodic)
    BIC_base = -2*ll_base+2*np.log(len(t))
    BIC_periodic = -2*ll_periodic+4*np.log(len(t))
    #print(BIC_base,BIC_periodic)
    res = (BIC_base-BIC_periodic)/2
    return theta_periodic,np.sqrt(theta_periodic[0][0]**2+theta_periodic[1][0]**2),res

## Calculate periodogram using history data and updated data

In [ ]:
P_list = np.exp(np.linspace(np.log(10),np.log(10000),2000))
lnBF_list = []
Amp_list = []
theta_list = []
for P in P_list:
    res_all = lnBF(P,(T_mid-np.min(T_mid)),O_C_ephemeride/60,e_T_mid*24*60)
    Amp_list.append(res_all[1])
    lnBF_list.append(np.squeeze(res_all[2]))
    theta_list.append(res_all[0])
new_lnBF_list = []
new_Amp_list = []
new_theta_list = []
for P in P_list:
    res_all = lnBF(P,(np.append(T_mid,time_barycentre)-np.min(T_mid)),np.append(O_C_ephemeride,O_C_this)/60,np.append(e_T_mid,e_t_m_median_jd_UTC)*24*60)
    new_Amp_list.append(res_all[1])
    new_lnBF_list.append(np.squeeze(res_all[2]))
    new_theta_list.append(res_all[0])
lnBF_list = np.array(lnBF_list)
new_lnBF_list = np.array(new_lnBF_list)
Amp_list = np.array(Amp_list)
new_Amp_list = np.array(new_Amp_list)

In [ ]:
# Show the results
plt.plot(P_list,lnBF_list,label = 'history data',alpha = 0.5)
plt.plot(P_list,new_lnBF_list,label = 'updated data',alpha = 0.5)
plt.plot([np.min(P_list),np.max(P_list)],[5,5],'--',label = 'lnBF = 5')

plt.ylim([-10,10])
plt.xscale('log')
plt.xlabel('P [d]')
plt.ylabel('lnBF')
plt.legend()

## Show the result of folded O-C with significant period

In [ ]:
P_best_list = P_list[new_lnBF_list>5]
theta_best_list = np.squeeze(np.array(new_theta_list)[new_lnBF_list>5])
for i in range(len(P_best_list)):
    P_best = P_best_list[i]
    theta_best = theta_best_list[i]
    y_show = (O_C_ephemeride/60-(T_mid-np.min(T_mid))*theta_best[3]-theta_best[2])*60
    plt.figure()
    plt.errorbar(((T_mid-np.min(T_mid))%P_best)/P_best,y_show,yerr = e_T_mid*24*3600, marker = '.',ls='none',label = 'Detrended history data')
    plt.errorbar([((time_barycentre-np.min(T_mid))%P_best)/P_best],[(O_C_this/60-(time_barycentre-np.min(T_mid))*theta_best[3]-theta_best[2])*60],yerr = e_t_m_median_jd_UTC*24*3600, marker = '.',ls='none',label = 'Detrended this measurement')
    t_range = np.linspace(0,1,200)
    Y_fit = theta_best[0]*np.cos(2*np.pi*t_range)+theta_best[1]*np.sin(2*np.pi*t_range)
    plt.title('P = '+str(P_best)+'day, Amplitude = '+str(60*np.sqrt(theta_best[0]**2+theta_best[1]**2))+'s')
    plt.ylabel('O-C [s]')
    plt.xlabel('phase')
    plt.plot(t_range,Y_fit*60,'--',label='best_fit')

# Question: are these signals likely to be caused by an undetected planet? Why or why not?